# SOC Triage Gym v2 - Trained Model Evaluation Notebook

This notebook runs end-to-end in Colab/local and evaluates your trained Tier-1 model on SOC-Triage-Gym v2 team tasks.

Outputs:
- Per-episode scores for baseline (oracle) and your trained model
- Average score comparison
- Evaluation graph saved as `soc_trained_eval_graph.png`



In [ ]:
# 1) Setup dependencies and repo path
import os
import sys
import subprocess
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn', 'httpx', 'matplotlib', 'numpy', 'torch', 'transformers', 'peft', 'accelerate'])

if not Path('server').exists() and not Path('openenv-2').exists():
    subprocess.check_call(['git', 'clone', '-q', 'https://github.com/ROHITCRAFTSYT/-Metas-OpenEnv-2.git', 'openenv-2'])

if Path('openenv-2').exists() and not Path('server').exists():
    os.chdir('openenv-2')

print('Working directory:', os.getcwd())



In [ ]:
# 2) Config: point this to your trained adapter/model
# If you trained in this repo, common path is ./soc_grpo_tier1
MODEL_PATH = os.environ.get('SOC_TRAINED_MODEL_PATH', './soc_grpo_tier1')
ROLE = 'tier1'
SEEDS = [100, 101, 102, 103, 104]
TASKS = ['team_phishing_escalation', 'team_lateral_team']

print('MODEL_PATH =', MODEL_PATH)



In [ ]:
# 3) Start SOC server
import time
import httpx
import subprocess

server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '127.0.0.1', '--port', '7860', '--log-level', 'warning'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

client = httpx.Client(base_url='http://127.0.0.1:7860', timeout=10.0)
ready = False
for _ in range(30):
    time.sleep(1)
    if server_proc.poll() is not None:
        raise RuntimeError(f'Server exited early with code {server_proc.returncode}')
    try:
        r = client.get('/health')
        if r.status_code == 200:
            ready = True
            print('Server ready:', r.json())
            break
    except Exception:
        pass

if not ready:
    server_proc.terminate()
    raise RuntimeError('Server did not become healthy in time')



In [ ]:
# 4) Load evaluation helpers and trained model (if present)
import json
import torch
from pathlib import Path

sys.path.insert(0, '.')
from train_grpo import run_episode, oracle_action, parse_action_from_text, format_obs_prompt, ROLE_SYSTEM_PROMPTS

tokenizer = None
model = None
use_trained_model = False

if Path(MODEL_PATH).exists():
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Try loading as full model; if that fails, load as adapter on a base model.
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype='auto', device_map='auto', trust_remote_code=True)
        use_trained_model = True
        print('Loaded full model from', MODEL_PATH)
    except Exception:
        base_model_name = os.environ.get('SOC_BASE_MODEL', 'Qwen/Qwen2.5-0.5B')
        base = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype='auto', device_map='auto', trust_remote_code=True)
        model = PeftModel.from_pretrained(base, MODEL_PATH)
        use_trained_model = True
        print('Loaded PEFT adapter from', MODEL_PATH, 'on base', base_model_name)
else:
    print('[WARN] Trained model path not found. Evaluation will fall back to oracle for trained role.')


def model_action(obs, role='tier1', step=0):
    if not use_trained_model:
        return oracle_action(obs)
    messages = [
        {'role': 'system', 'content': ROLE_SYSTEM_PROMPTS[role]},
        {'role': 'user', 'content': format_obs_prompt(obs, role, step)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    act = parse_action_from_text(text, role)
    return act if isinstance(act, dict) and 'action_type' in act else {'action_type': 'noop', 'role': role}



In [ ]:
# 5) Evaluate baseline vs trained-model policy

def run_policy_episode(task_id, seed, policy='oracle', role='tier1', max_steps=80):
    obs = client.post('/reset', json={'task_id': task_id, 'seed': seed, 'mode': 'team'}).json()
    step = 0
    while not obs.get('done', False) and step < max_steps:
        step += 1
        acting_role = obs.get('current_role', 'tier1')
        if policy == 'trained' and acting_role == role:
            action = model_action(obs, role=role, step=step)
        else:
            action = oracle_action(obs)
        obs = client.post('/step', content=json.dumps(action), headers={'Content-Type': 'application/json'}).json()
    return float(obs.get('task_score') or obs.get('cumulative_reward', 0.0))

baseline_scores = []
trained_scores = []

for task in TASKS:
    for seed in SEEDS:
        b = max(0.0, run_policy_episode(task, seed, policy='oracle', role=ROLE))
        t = max(0.0, run_policy_episode(task, seed, policy='trained', role=ROLE))
        baseline_scores.append({'task': task, 'seed': seed, 'score': b})
        trained_scores.append({'task': task, 'seed': seed, 'score': t})
        print(f'{task} seed={seed} | baseline={b:.4f} | trained={t:.4f}')

baseline_avg = sum(x['score'] for x in baseline_scores) / len(baseline_scores)
trained_avg = sum(x['score'] for x in trained_scores) / len(trained_scores)

print('
Baseline avg:', round(baseline_avg, 4))
print('Trained avg :', round(trained_avg, 4))
if baseline_avg > 0:
    print('Delta       :', round(trained_avg - baseline_avg, 4), f'({(trained_avg / baseline_avg - 1) * 100:+.2f}%)')



In [ ]:
# 6) Plot and save graph
import numpy as np
import matplotlib.pyplot as plt

b_vals = [x['score'] for x in baseline_scores]
t_vals = [x['score'] for x in trained_scores]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(['Baseline (oracle)', f'Trained ({ROLE})'], [baseline_avg, trained_avg], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Average Episode Score')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, max(1.0, max(baseline_avg, trained_avg) * 1.2))

x = np.arange(1, len(b_vals) + 1)
axes[1].plot(x, b_vals, 'o-', alpha=0.6, label='Baseline')
axes[1].plot(x, t_vals, 's-', alpha=0.8, label='Trained')
if len(t_vals) >= 3:
    w = min(5, len(t_vals))
    sm = np.convolve(t_vals, np.ones(w)/w, mode='valid')
    axes[1].plot(np.arange(w, len(t_vals)+1), sm, '-', linewidth=2.2, label='Trained (smoothed)')
axes[1].set_title('Per-Episode Scores')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].set_ylim(0, max(1.0, max(max(b_vals), max(t_vals)) * 1.1 if b_vals and t_vals else 1.0))

plt.tight_layout()
plt.savefig('soc_trained_eval_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved graph: soc_trained_eval_graph.png')



In [ ]:
# 7) Cleanup
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except Exception:
        server_proc.kill()
print('Done')

